In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv('../data/raw/aa_dataset-tickets-multi-lang-5-2-50-version.csv')
en = df[df['language'] == 'en'].copy()
en = en[['subject', 'body', 'type', 'queue', 'priority']].dropna(subset=['body', 'type', 'queue', 'priority'])
print(en.shape)

(16338, 5)


In [3]:
import re

def clean_text(text):
    text = text.lower()
    text = text.replace('\\n', ' ').replace('\\r', ' ').replace('\\t', ' ')  # strip literal escape sequences
    text = re.sub(r'[^a-z\s]', ' ', text)  # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()  # collapse whitespace
    return text

en['body_clean'] = en['body'].apply(clean_text)
en[['body', 'body_clean']].head(3)

,body,body_clean
1,"Dear Customer Support Team,\n\nI am writing to...",dear customer support team i am writing to rep...
2,"Dear Customer Support Team,\n\nI hope this mes...",dear customer support team i hope this message...
3,"Dear Customer Support Team,\n\nI hope this mes...",dear customer support team i hope this message...


In [4]:
X = en[['body_clean', 'type', 'queue']]
y = en['priority']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Train: (13070, 3), Test: (3268, 3)
priority
medium    0.405050
high      0.388447
low       0.206503
Name: proportion, dtype: float64
priority
medium    0.405141
high      0.388311
low       0.206548
Name: proportion, dtype: float64


In [5]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# One-hot encode queue + type
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_train_cat = ohe.fit_transform(X_train[['type', 'queue']])
X_test_cat = ohe.transform(X_test[['type', 'queue']])

# TF-IDF the cleaned text
tfidf = TfidfVectorizer(max_features=3000, stop_words='english', min_df=3)
X_train_text = tfidf.fit_transform(X_train['body_clean'])
X_test_text = tfidf.transform(X_test['body_clean'])

print("Categorical shape:", X_train_cat.shape)
print("Text shape:", X_train_text.shape)

Categorical shape: (13070, 14)
Text shape: (13070, 3000)


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Model A: categorical features only
clf_a = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
clf_a.fit(X_train_cat, y_train)
pred_a = clf_a.predict(X_test_cat)

print("=== MODEL A: queue + type only ===")
print("Accuracy:", accuracy_score(y_test, pred_a))
print(classification_report(y_test, pred_a))

=== MODEL A: queue + type only ===
Accuracy: 0.4923500611995104
              precision    recall  f1-score   support

        high       0.57      0.65      0.61      1269
         low       0.36      0.48      0.41       675
      medium       0.50      0.35      0.41      1324

    accuracy                           0.49      3268
   macro avg       0.48      0.49      0.48      3268
weighted avg       0.50      0.49      0.49      3268



In [7]:
# Model B: categorical + text combined
X_train_combined = hstack([X_train_cat, X_train_text])
X_test_combined = hstack([X_test_cat, X_test_text])

clf_b = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
clf_b.fit(X_train_combined, y_train)
pred_b = clf_b.predict(X_test_combined)

print("=== MODEL B: queue + type + text ===")
print("Accuracy:", accuracy_score(y_test, pred_b))
print(classification_report(y_test, pred_b))

=== MODEL B: queue + type + text ===
Accuracy: 0.7946756425948592
              precision    recall  f1-score   support

        high       0.80      0.84      0.82      1269
         low       0.91      0.62      0.74       675
      medium       0.75      0.84      0.79      1324

    accuracy                           0.79      3268
   macro avg       0.82      0.77      0.78      3268
weighted avg       0.80      0.79      0.79      3268



In [8]:
# Clean subject the same way as body
en['subject_clean'] = en['subject'].fillna('').apply(clean_text)

# Re-split to include subject_clean (or add it to your existing X_train/X_test if you already have the index)
X = en[['subject_clean', 'body_clean', 'type', 'queue']]
y = en['priority']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF for subject (smaller vocab since subjects are short)
tfidf_subject = TfidfVectorizer(max_features=500, stop_words='english', min_df=3)
X_train_subject = tfidf_subject.fit_transform(X_train['subject_clean'])
X_test_subject = tfidf_subject.transform(X_test['subject_clean'])

print("Subject TF-IDF shape:", X_train_subject.shape)

Subject TF-IDF shape: (13070, 500)


In [9]:
# Categorical (queue + type)
X_train_cat = ohe.fit_transform(X_train[['type', 'queue']])
X_test_cat = ohe.transform(X_test[['type', 'queue']])

# Body TF-IDF
X_train_text = tfidf.fit_transform(X_train['body_clean'])
X_test_text = tfidf.transform(X_test['body_clean'])

# Combine ALL features: categorical + body text + subject text
X_train_full = hstack([X_train_cat, X_train_text, X_train_subject])
X_test_full = hstack([X_test_cat, X_test_text, X_test_subject])

print("Full combined shape:", X_train_full.shape)

Full combined shape: (13070, 3514)


In [10]:
clf_c = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
clf_c.fit(X_train_full, y_train)
pred_c = clf_c.predict(X_test_full)

print("=== MODEL C: queue + type + body + subject ===")
print("Accuracy:", accuracy_score(y_test, pred_c))
print(classification_report(y_test, pred_c))

=== MODEL C: queue + type + body + subject ===
Accuracy: 0.7925336597307222
              precision    recall  f1-score   support

        high       0.80      0.84      0.82      1269
         low       0.93      0.61      0.74       675
      medium       0.75      0.84      0.79      1324

    accuracy                           0.79      3268
   macro avg       0.83      0.76      0.78      3268
weighted avg       0.80      0.79      0.79      3268



In [11]:
tfidf_v2 = TfidfVectorizer(max_features=5000, stop_words='english', min_df=3, ngram_range=(1,2))
X_train_text_v2 = tfidf_v2.fit_transform(X_train['body_clean'])
X_test_text_v2 = tfidf_v2.transform(X_test['body_clean'])

X_train_v2 = hstack([X_train_cat, X_train_text_v2])
X_test_v2 = hstack([X_test_cat, X_test_text_v2])

print("New combined shape:", X_train_v2.shape)

New combined shape: (13070, 5014)


In [12]:
clf_d = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
clf_d.fit(X_train_v2, y_train)
pred_d = clf_d.predict(X_test_v2)

print("=== MODEL D: RF with bigrams + 5000 features ===")
print("Accuracy:", accuracy_score(y_test, pred_d))
print(classification_report(y_test, pred_d))

=== MODEL D: RF with bigrams + 5000 features ===
Accuracy: 0.7888616891064871
              precision    recall  f1-score   support

        high       0.79      0.84      0.81      1269
         low       0.90      0.63      0.74       675
      medium       0.75      0.83      0.78      1324

    accuracy                           0.79      3268
   macro avg       0.81      0.76      0.78      3268
weighted avg       0.80      0.79      0.79      3268



In [13]:
from sklearn.linear_model import LogisticRegression

clf_e = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
clf_e.fit(X_train_v2, y_train)
pred_e = clf_e.predict(X_test_v2)

print("=== MODEL E: Logistic Regression with bigrams + 5000 features ===")
print("Accuracy:", accuracy_score(y_test, pred_e))
print(classification_report(y_test, pred_e))

=== MODEL E: Logistic Regression with bigrams + 5000 features ===
Accuracy: 0.6015911872705019
              precision    recall  f1-score   support

        high       0.66      0.66      0.66      1269
         low       0.49      0.64      0.55       675
      medium       0.63      0.52      0.57      1324

    accuracy                           0.60      3268
   macro avg       0.59      0.61      0.59      3268
weighted avg       0.61      0.60      0.60      3268



In [14]:
import joblib
import os

os.makedirs('../src/models', exist_ok=True)

joblib.dump(clf_b, '../src/models/priority_classifier.joblib')
joblib.dump(ohe, '../src/models/queue_type_encoder.joblib')
joblib.dump(tfidf, '../src/models/body_tfidf_vectorizer.joblib')

print("Saved model and vectorizers to src/models/")

Saved model and vectorizers to src/models/
